In [1]:
# Load FOLDOC.txt into variable 'corpus'
from pathlib import Path
corpus_path = Path("FOLDOC.txt")
# Try current notebook directory if not found
if not corpus_path.exists():
    corpus_path = Path.cwd() / "FOLDOC.txt"
with corpus_path.open("r", encoding="utf-8") as f:
    corpus = f.read()

# Basic check
print(f"Loaded corpus: {len(corpus)} characters")

Loaded corpus: 5452365 characters


In [2]:
from transformers import BertTokenizer


tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokens = tokenizer.tokenize(corpus)
print(len(tokens))


c:\Users\brtoone\AppData\Local\miniconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1386589


In [3]:
# Convert `tokens` (list) to sequential IDs: token -> 0,1,2,...
# Place this cell after the tokenizer cell. Creates `token_to_id` and `id_to_token`.
try:
    tokens
except NameError:
    raise NameError("`tokens` not found. Run the tokenizer cell first.")

# Preserve first-occurrence order and assign sequential IDs
unique_tokens = list(dict.fromkeys(tokens))
token_to_id = {tok: i for i, tok in enumerate(unique_tokens)}
id_to_token = {i: tok for tok, i in token_to_id.items()}

print(f"Assigned {len(token_to_id)} token IDs")
print("First 20 token->id entries:")
for tok, idx in list(token_to_id.items())[:20]:
    print(f"{tok!r}: {idx}")

# Keep variables in the notebook namespace for later use


Assigned 17898 token IDs
First 20 token->id entries:
'free': 0
'on': 1
'-': 2
'line': 3
'dictionary': 4
'of': 5
'computing': 6
'<': 7
'introduction': 8
'>': 9
'fold': 10
'##oc': 11
'is': 12
'a': 13
'search': 14
'##able': 15
'acronym': 16
'##s': 17
',': 18
'jar': 19


In [4]:
#print(token_to_id["helicopter"])
print(token_to_id["jet"])
print(token_to_id["pizza"])
print(token_to_id["airplane"])
print(token_to_id["plane"])


8460
15895
4671
5379


In [35]:
import gensim.downloader
glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')
#glove_vectors = gensim.downloader.load('glove-twitter-50')
#glove_vectors = gensim.downloader.load('glove-twitter-100')


In [42]:
import torch
import torch.nn as nn
from torch_kmeans import KMeans
import numpy as np

weights = torch.FloatTensor(np.array([glove_vectors.vectors[:10000]]))    
model = KMeans(n_clusters=5, max_iter=1000)
print(model)


KMeans(init: 'rnd', num_init: 8, max_iter: 1000, distance: LpDistance(), tolerance: 0.0001, normalize: None)


In [43]:

result = model(weights)


Full batch converged at iteration 153/1000 with center shifts = tensor([0.]).


In [44]:
print(glove_vectors.index_to_key[55])
print(len(weights[0]))
for i, element in enumerate(result.labels[0]):
    cluster = element.item()
    print(glove_vectors.index_to_key[i], cluster)

two
10000
the 3
, 2
. 3
of 3
to 3
and 3
in 3
a 3
" 3
's 3
for 3
- 1
that 3
on 1
is 3
was 2
said 4
with 3
he 3
as 3
it 3
by 3
at 1
( 2
) 2
from 1
his 3
'' 3
`` 3
an 3
be 3
has 3
are 3
have 3
but 3
were 1
not 3
this 3
who 2
they 3
had 4
i 3
which 3
will 3
their 3
: 3
or 3
its 3
one 3
after 4
new 3
been 3
also 3
we 3
would 3
two 1
more 3
' 3
first 2
about 3
up 1
when 1
year 0
there 3
all 3
-- 3
out 1
she 3
other 3
people 3
n't 3
her 3
percent 0
than 0
over 4
into 1
last 4
some 3
government 4
time 3
$ 0
you 3
years 3
if 3
no 3
world 3
can 3
three 1
do 3
; 2
president 4
only 3
state 4
million 0
could 3
us 0
most 3
_ 3
against 4
u.s. 4
so 3
them 3
what 3
him 3
united 4
during 4
before 1
may 3
since 4
many 3
while 3
where 1
states 4
because 3
now 3
city 1
made 3
like 3
between 3
did 3
just 1
national 4
day 1
country 4
under 4
such 3
second 2
then 1
company 0
group 4
any 3
through 1
china 4
four 1
being 3
down 1
war 4
back 1
off 1
south 4
american 3
minister 4
police 4
well 3
including 3
team 

In [122]:
import torch.nn as nn
import numpy as np
import torch

# let's compare the word embeddings for airplane, jet, helicopter, and pizza
input_strs = ["airplane tuning airplane", "jogging flying running walking skipping", "horse mule donkey ox goat sheep cow", "raven crow parrot parakeet robin sandwich airplane"]
word_embeddings = [[(w,glove_vectors[w]) for w in input_str.split()] for input_str in input_strs]
#position_encodings_for_input_str = pe(word_embeddings_for_input_str)
#print(word_embeddings_for_input_str)
#print(position_encodings_for_input_str)

# calculate the similarity between each word embedding
for prompt_we in word_embeddings:
    base_we = prompt_we[0]
    position_encoded_values = pe(torch.tensor([we for (_, we) in prompt_we]))
    print(position_encoded_values)
   # print(base_we[1])
   # print(base_we[1])
    diffs = []
    for (w, we) in prompt_we[1:]:
        thediffs = base_we[1] - we
        buckets = []
        inc = 0.05
        while inc <= 0.5:
            buckets.append([round(inc,2), 0])
            inc += 0.05
        buckets.append([999,0])
        for thediff in thediffs:
            for i, bucket in enumerate(buckets):
                if thediff<bucket[0]:
                    buckets[i][1] += 1
                    break

        diff = np.sum(np.absolute(thediffs))
        diffs.append((w, diff))
        print(base_we[0],w,diff)
        #print(buckets)
    print()



#position_encoded1 = pe(word_embeddings1)
#print(sample_input0)
#print(sample_input1)
#print(word_embeddings0[12])
#print(word_embeddings1[6])
#print(position_encoded0)
#print(position_encoded1)


tensor([[-0.8102,  1.0659,  0.0373,  1.2097, -1.0245,  1.1300,  0.8967,  0.1079,
          0.9043,  0.8233,  0.1047,  0.9403, -2.1969,  1.0455,  0.5513,  1.0379,
         -0.1607,  0.7650,  0.1055,  1.0360, -0.1099,  0.4596,  0.5339,  0.5242,
         -0.2936,  1.6722, -1.2894,  0.9803, -0.2439,  1.3535, -0.2215,  1.0998,
          0.4103, -0.2657,  0.6741,  1.8334,  0.4900,  1.1284,  0.6846,  0.6808,
          0.4027,  1.7423, -0.7163,  0.0175,  0.2515,  1.9001, -0.2640,  1.3016,
          0.0097,  2.3439],
        [ 0.1039,  1.2650,  0.6589,  0.4568,  0.6528,  1.1569,  0.7743,  0.3696,
          1.0471,  0.3078,  0.3246,  2.0295, -1.6957,  1.0689, -0.3962,  0.9283,
          0.2478,  1.3198, -0.0954,  1.6729,  0.0093,  0.9678,  0.2883,  0.6971,
         -0.2878,  1.6280, -0.1053,  1.6257, -0.3253,  1.4114,  0.1101,  0.5853,
          0.8199,  0.8883,  0.8771,  0.8740,  0.0637,  0.7161, -0.3123,  2.1324,
          0.7133,  1.9380, -0.0726,  1.5646, -0.0839,  1.2420, -0.4476,  1.3960,


In [ ]:

#from sklearn.manifold import TSNE

#x_np = word_embeddings0.numpy()
#x3d = TSNE(n_components=3, perplexity=30, learning_rate=200).fit_transform(x_np)

#proj = torch.nn.Linear(512, 3)
#x3d = proj(word_embeddings0).detach().numpy()
#print(len(x3d))

import matplotlib.pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x3d[:,0], x3d[:,1], x3d[:,2])
plt.show()
